# 03. X Feature 집계
- 02_preprocessing.ipynb 완료 후 실행
- 순서: 구종 그룹화 → X구간 feature 집계 → delta feature 적용 → 최종 feature table 저장

In [ ]:
# ── 환경 감지 ──────────────────────────────────────────────
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MLB_pitcher/data'
else:
    BASE_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

INTERIM_DIR  = os.path.join(BASE_DIR, 'interim')
FEATURE_DIR  = os.path.join(BASE_DIR, 'features')
os.makedirs(FEATURE_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'INTERIM : {INTERIM_DIR}')
print(f'FEATURE : {FEATURE_DIR}')

In [ ]:
# ── DuckDB 설치 ────────────────────────────────────────────
try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'duckdb', '-q'])
    import duckdb

import pandas as pd
import numpy as np

print(f'DuckDB version: {duckdb.__version__}')

## 1. DuckDB 적재

In [ ]:
# ── DuckDB 연결 및 parquet 뷰 등록 ────────────────────────
DB_PATH = os.path.join(BASE_DIR, 'mlb_pitcher.duckdb')
con = duckdb.connect(DB_PATH)

# 선발투수 전처리 결과 parquet → DuckDB 테이블
starters_path = os.path.join(INTERIM_DIR, 'starters_all.parquet')
lookup_path   = os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet')

con.execute(f"""
    CREATE OR REPLACE TABLE starters AS
    SELECT * FROM read_parquet('{starters_path}')
""")

con.execute(f"""
    CREATE OR REPLACE TABLE prev_lookup AS
    SELECT * FROM read_parquet('{lookup_path}')
""")

n_rows = con.execute('SELECT COUNT(*) FROM starters').fetchone()[0]
print(f'starters 테이블: {n_rows:,}행 적재 완료')

In [ ]:
# ── 구종 그룹 컬럼 추가 ────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE starters AS
    SELECT *,
        CASE pitch_type
            WHEN 'FF' THEN 'Fastball'
            WHEN 'SI' THEN 'Fastball'
            WHEN 'FC' THEN 'Fastball'
            WHEN 'SL' THEN 'Breaking'
            WHEN 'CU' THEN 'Breaking'
            WHEN 'KC' THEN 'Breaking'
            WHEN 'CH' THEN 'Offspeed'
            WHEN 'FS' THEN 'Offspeed'
            ELSE 'Other'
        END AS pitch_group,
        CASE
            WHEN description IN ('called_strike','swinging_strike',
                                  'swinging_strike_blocked','foul',
                                  'foul_tip') THEN 1
            ELSE 0
        END AS is_strike
    FROM starters
""")
print('구종 그룹 + strike 컬럼 추가 완료')

## 2. X 구간 설정

In [ ]:
# 실험할 구간 조합 — 나중에 X 구간 실험(04)에서 전부 비교
# 여기서는 우선 batter/9 로 feature 생성
MODE = 'batter'   # 'pitch' | 'inning' | 'batter'
N    = 9

# X구간 필터 SQL 조건
if MODE == 'pitch':
    # 투구 순서 번호가 없으므로 pitch_number 기준 근사
    x_filter = f'pitch_number <= {N}'
elif MODE == 'inning':
    x_filter = f'inning <= {N}'
elif MODE == 'batter':
    # at_bat_number 기준: 경기별 최솟값 + N - 1
    x_filter = f'at_bat_number <= min_ab + {N} - 1'

print(f'X 구간 설정: mode={MODE}, n={N}')

## 3. X Feature 집계 (SQL)

In [ ]:
# ── batter 모드: 경기별 at_bat_number 최솟값 먼저 계산 ────
con.execute("""
    CREATE OR REPLACE TABLE game_ab_min AS
    SELECT game_pk, pitcher, MIN(at_bat_number) AS min_ab
    FROM starters
    GROUP BY game_pk, pitcher
""")

# X구간 데이터 뷰 생성
con.execute("""
    CREATE OR REPLACE VIEW x_zone AS
    SELECT s.*
    FROM starters s
    JOIN game_ab_min g
      ON s.game_pk = g.game_pk AND s.pitcher = g.pitcher
    WHERE s.at_bat_number <= g.min_ab + 9 - 1
""")

n_x = con.execute('SELECT COUNT(*) FROM x_zone').fetchone()[0]
print(f'X구간 투구 수: {n_x:,}행')

In [ ]:
# ── 구종별 feature 집계 ────────────────────────────────────
# 구종 그룹별 (Fastball / Breaking / Offspeed) 평균 구속, 구속 분산, spin rate
pitch_group_features = con.execute("""
    SELECT
        game_pk,
        pitcher,
        season,
        pitch_group,
        COUNT(*)                          AS pitch_count,
        AVG(release_speed)                AS avg_speed,
        STDDEV(release_speed)             AS std_speed,
        AVG(release_spin_rate)            AS avg_spin,
        AVG(release_extension)            AS avg_ext,
        AVG(release_pos_x)                AS avg_pos_x,
        AVG(release_pos_z)                AS avg_pos_z
    FROM x_zone
    WHERE pitch_group != 'Other'
    GROUP BY game_pk, pitcher, season, pitch_group
""").df()

print(f'구종별 집계: {len(pitch_group_features):,}행')
pitch_group_features.head()

In [ ]:
# ── 경기 단위 전체 feature 집계 ────────────────────────────
game_features = con.execute("""
    SELECT
        game_pk,
        pitcher,
        season,
        -- 전체 투구 수
        COUNT(*)                                        AS total_pitches,
        -- 전체 평균 구속 / 분산
        AVG(release_speed)                              AS avg_speed_all,
        STDDEV(release_speed)                           AS std_speed_all,
        -- Strike ratio
        AVG(is_strike)                                  AS strike_ratio,
        -- 구종 비율
        AVG(CASE WHEN pitch_group = 'Fastball' THEN 1.0 ELSE 0.0 END) AS fastball_ratio,
        AVG(CASE WHEN pitch_group = 'Breaking' THEN 1.0 ELSE 0.0 END) AS breaking_ratio,
        AVG(CASE WHEN pitch_group = 'Offspeed' THEN 1.0 ELSE 0.0 END) AS offspeed_ratio,
        -- 릴리스 포인트
        AVG(release_pos_x)                              AS avg_pos_x,
        AVG(release_pos_z)                              AS avg_pos_z,
        AVG(release_extension)                          AS avg_ext,
        -- arm angle
        AVG(arm_angle)                                  AS avg_arm_angle
    FROM x_zone
    GROUP BY game_pk, pitcher, season
""").df()

print(f'경기 단위 feature: {len(game_features):,}행')
game_features.head()

In [ ]:
# ── 구종별 feature → wide format으로 pivot ────────────────
pivot = pitch_group_features.pivot_table(
    index=['game_pk', 'pitcher', 'season'],
    columns='pitch_group',
    values=['avg_speed', 'std_speed', 'avg_spin', 'avg_ext'],
    aggfunc='first'
)
# 컬럼명 평탄화: avg_speed_Fastball 형태
pivot.columns = [f'{v}_{g}' for v, g in pivot.columns]
pivot = pivot.reset_index()

# 경기 단위 feature와 병합
features = game_features.merge(pivot, on=['game_pk', 'pitcher', 'season'], how='left')
print(f'병합 후: {len(features):,}행  |  컬럼 수: {len(features.columns)}')
print(features.columns.tolist())

## 4. Delta Feature 적용 (직전 시즌 대비 편차)

In [ ]:
# lookup table 로드
lookup = pd.read_parquet(os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet'))

# 구종 그룹별 직전 시즌 평균 구속 pivot
lookup_pivot = lookup.pivot_table(
    index=['pitcher', 'season'],
    columns='pitch_type',
    values='prev_avg_speed',
    aggfunc='mean'
)
# 구종 그룹 기준으로 재집계 (Fastball 계열 평균 등)
FASTBALL = ['FF', 'SI', 'FC']
BREAKING = ['SL', 'CU', 'KC']
OFFSPEED = ['CH', 'FS']

existing = lookup_pivot.columns.tolist()
lookup_pivot['prev_speed_Fastball'] = lookup_pivot[[c for c in FASTBALL if c in existing]].mean(axis=1)
lookup_pivot['prev_speed_Breaking'] = lookup_pivot[[c for c in BREAKING if c in existing]].mean(axis=1)
lookup_pivot['prev_speed_Offspeed'] = lookup_pivot[[c for c in OFFSPEED if c in existing]].mean(axis=1)
lookup_pivot = lookup_pivot[['prev_speed_Fastball', 'prev_speed_Breaking', 'prev_speed_Offspeed']].reset_index()

# features 와 병합
features = features.merge(lookup_pivot, on=['pitcher', 'season'], how='left')

print('lookup 병합 완료')
lookup_pivot.head()

In [ ]:
# ── delta feature 계산 ─────────────────────────────────────
# 오늘 구속 - 직전 시즌 평균 구속 (구종 그룹별)
for group in ['Fastball', 'Breaking', 'Offspeed']:
    today_col = f'avg_speed_{group}'
    prev_col  = f'prev_speed_{group}'
    delta_col = f'delta_speed_{group}'

    if today_col in features.columns and prev_col in features.columns:
        features[delta_col] = features[today_col] - features[prev_col]
        print(f'{delta_col}: {features[delta_col].notna().sum():,}개 유효값')
    else:
        print(f'{delta_col}: 컬럼 없음 ({today_col} or {prev_col})')

# 2021 시즌은 직전 시즌(2020) 데이터 없으므로 delta = NaN → 정상
print(f'\n2021 시즌 delta NaN 비율: {features[features["season"]==2021]["delta_speed_Fastball"].isna().mean():.1%} (정상)')

## 5. Y값 병합 및 최종 저장

In [ ]:
# 02에서 계산한 Y값 로드
y_path = os.path.join(INTERIM_DIR, 'y_woba_batter9.parquet')
y_df   = pd.read_parquet(y_path)[['game_pk', 'pitcher', 'season', 'y_woba']]

# X feature + Y 병합
final = features.merge(y_df, on=['game_pk', 'pitcher', 'season'], how='inner')

# Y 결측 제거
final = final.dropna(subset=['y_woba'])

print(f'최종 데이터셋: {len(final):,}행  |  컬럼 수: {len(final.columns)}')
print(f'\ny_woba 분포:')
print(final['y_woba'].describe())

In [ ]:
# ── feature 목록 확인 ──────────────────────────────────────
meta_cols    = ['game_pk', 'pitcher', 'season', 'y_woba']
feature_cols = [c for c in final.columns if c not in meta_cols]

print(f'feature 수: {len(feature_cols)}')
for c in feature_cols:
    na_rate = final[c].isna().mean()
    print(f'  {c:<35} NaN: {na_rate:.1%}')

In [ ]:
# ── 저장 ──────────────────────────────────────────────────
out = os.path.join(FEATURE_DIR, 'features_batter9.parquet')
final.to_parquet(out, index=False)
print(f'저장 완료 → {out}')

# DuckDB에도 등록
con.execute(f"""
    CREATE OR REPLACE TABLE features AS
    SELECT * FROM read_parquet('{out}')
""")
print('DuckDB features 테이블 등록 완료')

final.head()

In [ ]:
# ── 시즌별 샘플 수 확인 ────────────────────────────────────
con.execute("""
    SELECT season, COUNT(*) AS n_games
    FROM features
    GROUP BY season
    ORDER BY season
""").df()

In [ ]:
con.close()
print('DuckDB 연결 종료')